In [ ]:
# Estrutura de tabela de dispersão

guardaRoupas = {
    "divisao_1": "blusa",
    "divisao_2" : "suéter",
    "divisao_3" : "calça"
}

print(guardaRoupas)

exemploHash = hash("divisao_1")
print(exemploHash)

{'divisao_1': 'blusa', 'divisao_2': 'suéter', 'divisao_3': 'calça'}
8911013876926221972


In [26]:
# Coleta de dados meteorológicos

import csv
import requests

URL = "https://api.thingspeak.com/channels/12397/feeds.json"

LIMITE = {"results":8000}

resposta = requests.get(URL, params=LIMITE,timeout = 15)
resposta.raise_for_status()
dadosBrutos = resposta.json()

print(dadosBrutos)

informacoesDaEstacao = dadosBrutos["channel"]
leiturasDaEstacao = dadosBrutos["feeds"]

print("=== Informações de coleta ===")
print(f"Dados coletados: {len(leiturasDaEstacao)} da estação {informacoesDaEstacao.get('name')}")
print ("===        ===")

# Criando o dataset

caminhoDoArquivo = "dadosColetados.csv"

tabelaHash = {}
i = 0
for leitura in leiturasDaEstacao:
  entryID = leitura.get("entry_id")

  leituraBruta = 0.0
  chuva = 0.0

  if leitura.get("field5") is not None:
    leituraBruta = float(leitura.get("field5"))

  if (leituraBruta > 0.0):
    chuva = leituraBruta   

  tabelaHash[entryID] = {
      "id": i,
      "criado_em": leitura.get("created_at"),
      "direcao_vento": leitura.get("field1"),
      "velocidade_vento": leitura.get("field2"),
      "umidade": leitura.get("field3"),
      "temperatura": leitura.get("field4"),
      "intensidade_de_chuva": leitura.get("field5"),
      "pressao": leitura.get("field6"),
      "choveu": bool(chuva is not None and chuva > 0.0)
  }

  i+=1

print(tabelaHash)

campos = list(tabelaHash.values())[0].keys()
print(f"campos: {campos}")

with open(caminhoDoArquivo, "w", encoding="utf-8", newline="") as f:
  escritor = csv.DictWriter(f, fieldnames=campos)
  escritor.writeheader()
  for entryID, leitura in tabelaHash.items():
    escritor.writerow(leitura)

{'channel': {'id': 12397, 'name': 'WeatherStation', 'description': 'MathWorks Weather Station, West Garage, Natick, MA 01760, USA', 'latitude': '42.299676', 'longitude': '-71.350525', 'field1': 'Wind Direction (North = 0 degrees)', 'field2': 'Wind Speed (mph)', 'field3': '% Humidity', 'field4': 'Temperature (F)', 'field5': 'Rain (Inches/minute)', 'field6': 'Pressure ("Hg)', 'field7': 'Power Level (V)', 'field8': 'Light Intensity', 'created_at': '2014-05-20T21:50:32Z', 'updated_at': '2025-10-02T21:56:57Z', 'elevation': '60', 'last_entry_id': 5721756}, 'feeds': [{'created_at': '2026-08-21T09:16:27Z', 'entry_id': 5713757, 'field1': '175', 'field2': '1.3', 'field3': '0', 'field4': '0.1', 'field5': '0', 'field6': '29.83', 'field7': '0', 'field8': '0'}, {'created_at': '2026-08-21T09:17:27Z', 'entry_id': 5713758, 'field1': '122', 'field2': '1.6', 'field3': '0', 'field4': '0.1', 'field5': '0', 'field6': '29.83', 'field7': '0', 'field8': '0'}, {'created_at': '2026-08-21T09:18:27Z', 'entry_id': 

# Treinamento do Modelo 1 - Sem Balanceamento

In [ ]:
import pandas as pd
import numpy as np
# Carregando técnicas de IA/ML
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
# Carregando ferramentas de normalização
from sklearn.preprocessing import StandardScaler
# Carregando as métricas
from sklearn.metrics import average_precision_score, precision_recall_curve, f1_score

# Ferramentas de balanceamento/equilibrio do dataset
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN

# validação inicial, sem o balanceamento
caminhoCSV = "dadosColetados.csv"

df = pd.read_csv(caminhoCSV)
# display(df)

colunaAlvo = "choveu"
df[colunaAlvo] = df[colunaAlvo].astype(str).str.lower().map({"true":1, "false":0, "1": 1, "0":0})
df = df.sort_values("id").reset_index(drop=True)

print(f"Total de registros: {len(df)}")
print(f"Registros de chuva: {df[colunaAlvo].mean():.2%}")


colunasBase = ["direcao_vento", "velocidade_vento", "umidade", "temperatura"]

for coluna in colunasBase:
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce" )

df.dropna(subset=colunasBase)

# Validar a taxa de variação instantanea
for coluna in colunasBase:
    df[f"delta_{coluna}"] = df[coluna].diff().fillna(0)

for coluna in colunasBase:
    df[f"media_{coluna}"] = df[coluna].rolling(window=3).mean()

print(f"Total de colunas {len(df.columns)}")

# Preparando o ambiente e conjunto de dados
normalizador = StandardScaler()


Total de registros: 8000
Registros de chuva: 0.38%
Total de colunas 17


,id,criado_em,direcao_vento,velocidade_vento,umidade,temperatura,intensidade_de_chuva,pressao,choveu,delta_direcao_vento,delta_velocidade_vento,delta_umidade,delta_temperatura,media_direcao_vento,media_velocidade_vento,media_umidade,media_temperatura
0,0,2026-08-21T09:16:27Z,175,1.3,0,0.1,0.0,29.83,0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
1,1,2026-08-21T09:17:27Z,122,1.6,0,0.1,0.0,29.83,0,-53.0,0.3,0.0,0.0,NaN,NaN,NaN,NaN
2,2,2026-08-21T09:18:27Z,23,3.0,0,0.1,0.0,29.83,0,-99.0,1.4,0.0,0.0,106.666667,1.966667,0.0,0.1
3,3,2026-08-21T09:19:27Z,27,2.4,0,0.1,0.0,29.83,0,4.0,-0.6,0.0,0.0,57.333333,2.333333,0.0,0.1
4,4,2026-08-21T09:20:27Z,102,1.7,0,0.1,0.0,29.83,0,75.0,-0.7,0.0,0.0,50.666667,2.366667,0.0,0.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,7995,2026-08-27T00:24:34Z,0,1.9,0,0.1,0.0,29.84,0,0.0,-0.3,0.0,0.0,0.000000,1.966667,0.0,0.1
7996,7996,2026-08-27T00:25:34Z,0,2.6,0,0.1,0.0,29.84,0,0.0,0.7,0.0,0.0,0.000000,2.233333,0.0,0.1
7997,7997,2026-08-27T00:26:34Z,0,1.7,0,0.1,0.0,29.84,0,0.0,-0.9,0.0,0.0,0.000000,2.066667,0.0,0.1
7998,7998,2026-08-27T00:27:34Z,0,0.8,0,0.1,0.0,29.84,0,0.0,-0.9,0.0,0.0,0.000000,1.700000,0.0,0.1
